[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Properties


## What you will be able to do

Give a class attributes that are worked out when they are read, or checked when they are assigned,
while every piece of code that uses the class goes on writing plain `station.mean` and
`station.unit = "F"`.


## The idea

### The problem

A class usually begins with plain attributes, and that is the right way for it to begin. `__init__`
stores `self.unit = unit` and works out `self.mean` from the readings, and code all over the program
reads `station.mean` and assigns `station.unit = "F"`.

Then two requirements arrive. The mean has to stay current when a reading is added, and a stored
`self.mean` cannot: it was worked out once, in `__init__`, and nothing works it out again. And the
unit has to be checked, because `station.unit = "kelvin"` is stored like any other string, and every
conversion that later trusts it gives a wrong answer.

The fix most languages teach is a pair of methods. `get_mean()` works the mean out on every call,
and `set_unit()` checks before it stores. Both work, and both change how the attribute is written:
every `station.mean` in the program has to become `station.get_mean()`, and every
`station.unit = "F"` has to become `station.set_unit("F")`. An assignment that gets missed does not
fail. It creates a new attribute called `unit` that nothing reads, and the change it was meant to
make is lost. So those languages tell you to write a getter and a setter for every attribute from
the start, in case one is ever needed, which is the habit the **Why Classes** notebook argued
against.

Python needs neither. An attribute can begin plain and become calculated or checked later, and no
code that uses it has to change.

### What a property is

> A **property** is an attribute whose reading and assignment run methods. `@property` above a
> method named `mean` makes `station.mean` call that method and give back its result, with no
> parentheses. `@mean.setter` above a second method with the same name makes `station.mean = value`
> call that method instead of storing the value.

### Why it works that way

From outside, a property cannot be told apart from stored data, and that is the point of it. The
way callers write an attribute is the agreement between them and the class. What happens behind it
can change from a stored value to a calculation without any of them noticing.

A computed property is always current, because it is worked out at the moment it is read. The cost
is that it is worked out every time, and `functools.cached_property`, near the end, trades that back
for a value that can go stale.

A setter sees every assignment, so a check placed there cannot be skipped by code that forgot it
existed. The setter needs somewhere to keep the value other than the property itself, and by
convention that place is the same name with a leading underscore, `self._unit`. The underscore tells
other programmers the value is internal. Python does not enforce it.

A property with no setter cannot be assigned at all, which is exactly what a calculated value like
`mean` should do.

And `@property` is a decorator, from the **Decorators** notebook: it means `mean = property(mean)`.
This notebook takes apart what that builds, and what `@mean.setter` adds to it.

### Where you will meet this

`Path("data/readings.csv").suffix` and `.parent` are properties, which is why neither takes
parentheses. In the **Pandas** guide, `df.shape` and `df.columns` work the same way.

### What this notebook covers

The same class written three ways, with plain attributes, with getter and setter methods, and with
properties, each run through the same calling code. Then a computed attribute, read-only by
default, checks on the way in and in `__init__`, the underscore convention, a property that
converts, how `@property` is built, and `cached_property`. Then one class that uses all of it.

### A first look

A property that works out a mean. There is nothing to run yet: read it, and read the output
underneath it.

```python
class Station:
    def __init__(self, readings):
        self.readings = readings

    @property
    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)


station = Station([-4.1, -2.6])
print(station.mean)

station.readings.append(-5.2)
print(station.mean)
```

```
-3.35
-3.97
```

No parentheses, and the second value is current, because it was worked out when it was read.


## Setup

One import.

- `functools` provides `cached_property`, which is used in one section near the end

Every class in this notebook is written in the section that uses it, so this cell only imports.

**Run this cell before the rest of the notebook.**


In [1]:
import functools

print("ready")


ready


## Worked examples

### Before and after: one class, three ways

Here is the problem from the top of this notebook, in code. The calling code is written once, as a
function, and every version of the class is run through that same function, so no version gets
calling code written to suit it.


In [2]:
def session(station):
    """The calling code. Every version of the class is run through this same function."""
    print("  mean:                    ", station.mean)
    station.readings.append(-5.2)
    print("  mean after a new reading:", station.mean)
    station.unit = "F"
    print("  unit:                    ", station.unit)


First, plain attributes: the natural way to write the class, and the way it would usually start.


In [3]:
class PlainStation:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit
        self.mean = round(sum(readings) / len(readings), 2)


session(PlainStation("Tromso", [-4.1, -2.6]))

check = PlainStation("Tromso", [-4.1, -2.6])
check.unit = "kelvin"
print("  unit = 'kelvin' was stored:", check.unit)


  mean:                     -3.35
  mean after a new reading: -3.35
  unit:                     F
  unit = 'kelvin' was stored: kelvin


Two silent failures. The mean printed the same number twice, although a colder reading had been
added, because `self.mean` was worked out once and stored. And `'kelvin'` was stored as readily as
`'F'`.

Next, the fix most languages teach: methods that calculate and check.


In [4]:
class GetterStation:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.set_unit(unit)

    def get_mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def get_unit(self):
        return self._unit

    def set_unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value


getter = GetterStation("Tromso", [-4.1, -2.6])
getter.readings.append(-5.2)
print("  get_mean() after a new reading:", getter.get_mean())

try:
    getter.set_unit("kelvin")
except ValueError as error:
    print("  set_unit('kelvin'):", error)


  get_mean() after a new reading: -3.97
  set_unit('kelvin'): unit must be 'C' or 'F', not 'kelvin'


Both problems are fixed, for code that calls the methods. The calling code that already exists does
not call them. It is written as `station.mean` and `station.unit = ...`, and there is no longer any
attribute called `mean`.


In [5]:
try:
    session(GetterStation("Tromso", [-4.1, -2.6]))
except AttributeError as error:
    print("  AttributeError:", error)

missed = GetterStation("Tromso", [-4.1, -2.6])
missed.unit = "F"
print("  after missed.unit = 'F', get_unit() says:", repr(missed.get_unit()))
print("  and the stray attribute says:            ", repr(missed.unit))


  AttributeError: 'GetterStation' object has no attribute 'mean'
  after missed.unit = 'F', get_unit() says: 'C'
  and the stray attribute says:             'F'


The first call fails loudly, which is the better of the two outcomes. The second is worse:
`missed.unit = 'F'` created a new attribute called `unit`, which no method reads, and `get_unit()`
still reports `'C'`. Every assignment in the program that was not rewritten now does nothing, and
says nothing.

Finally, properties. The class changes; the calling code does not.


In [6]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    @property
    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value


session(Station("Tromso", [-4.1, -2.6]))

check = Station("Tromso", [-4.1, -2.6])
try:
    check.unit = "kelvin"
except ValueError as error:
    print("  unit = 'kelvin' was rejected:", error)


  mean:                     -3.35
  mean after a new reading: -3.97
  unit:                     F
  unit = 'kelvin' was rejected: unit must be 'C' or 'F', not 'kelvin'


The same `session` function, unchanged, and the mean is current this time. `'kelvin'` is rejected
by the assignment itself, on the line that tried it.

| | Plain attributes | Getter and setter methods | Properties |
|---|---|---|---|
| How calling code reads the mean | `station.mean` | `station.get_mean()` | `station.mean` |
| How calling code sets the unit | `station.unit = "F"` | `station.set_unit("F")` | `station.unit = "F"` |
| The mean after a new reading | stale | current | current |
| `unit = "kelvin"` | stored | rejected, but only through `set_unit` | rejected on assignment |
| Calling code already written | runs, and gets wrong answers | fails, or silently does nothing | runs, and gets right answers |

That is what a property is for: the class can change how an attribute works, and nothing that uses
the attribute has to know.

The rest of this notebook takes the third version apart.

| Question about the property version | The section that answers it |
|---|---|
| How does `station.mean` run a method without parentheses? | A computed attribute |
| What happens if something assigns to `mean`? | Read-only by default |
| How is `unit = "kelvin"` caught? | Checking a value on the way in |
| Why does `__init__` assign `self.unit`, not `self._unit`? | Checking in `__init__` too |
| Why is the value kept in `_unit`? | The underscore convention |
| What do `@property` and `@unit.setter` actually build? | How `@property` is built |

### A computed attribute

`@property` above a method makes the attribute of the same name run that method whenever it is
read. The method takes only `self`, because reading an attribute passes nothing else.


In [7]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @property
    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    @property
    def coldest(self):
        return min(self.readings)


north = Station("Tromso", [-4.1, -2.6])

print("mean:   ", north.mean)
print("coldest:", north.coldest)

north.readings.append(-9.9)
print("after a colder reading:", north.mean, north.coldest)

print()
print("on the class, mean is a", type(Station.mean).__name__)
print("on the object, it is a ", type(north.mean).__name__)


mean:    -3.35
coldest: -4.1
after a colder reading: -5.53 -9.9

on the class, mean is a property
on the object, it is a  float


Neither property stores anything. Both are worked out from `readings` each time, so a colder reading
changed both answers without anything being told to update.

On the class, `mean` is a `property` object. On an object, reading it runs the method and gives back
a plain `float`. That is the difference from the **Methods** notebook, where `north.average` without
parentheses gave back the method itself.

### Read-only by default

A property with only `@property` has no way to accept a value, so assigning to it raises.


In [8]:
try:
    north.mean = 0
except AttributeError as error:
    print("AttributeError:", error)


AttributeError: property 'mean' of 'Station' object has no setter


The message names exactly what is missing. For a calculated value this is the right behavior: a mean
assigned by hand would disagree with the readings the moment it was stored.

### Checking a value on the way in

`@unit.setter` above a second method, also named `unit`, gives the property a setter. Every
assignment to `station.unit` now calls it, with the assigned value as its argument. The check goes
in the setter, and so does the line that actually stores the value.


In [9]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value


north = Station("Tromso", [-4.1, -2.6])

north.unit = "F"
print("accepted:", north.unit)

try:
    north.unit = "kelvin"
except ValueError as error:
    print("rejected:", error)
print("still:   ", north.unit)


accepted: F
rejected: unit must be 'C' or 'F', not 'kelvin'
still:    F


The rejected assignment left the old value in place, because the check raised before the storing line
ran. The mistake is reported on the line that made it, not later, when a conversion goes wrong.

### Checking in `__init__` too

`__init__` assigns `self.unit = unit`, not `self._unit = unit`. That one character decides whether
construction is checked. Assigned through the property, a station cannot be created with a bad unit
either.


In [10]:
try:
    Station("Tromso", [-4.1, -2.6], unit="kelvin")
except ValueError as error:
    print("rejected at construction:", error)


rejected at construction: unit must be 'C' or 'F', not 'kelvin'


The quiet error at the end of this notebook shows the other spelling, and what it lets through.

### The underscore convention

The value lives in `_unit`. A single leading underscore is a convention that means internal: code
outside the class should use the property. It is only a convention.


In [11]:
north._unit = "kelvin"

print("north.unit is now:", north.unit)


north.unit is now: kelvin


Python did not stop that. Writing to `_unit` goes around the setter entirely.

The underscore does not make the check impossible to skip. It makes skipping it obviously
deliberate, and in practice that is enough: code that writes to another class's underscore
attributes is announcing that it knows it is doing something the class does not support.

### A property that converts

A setter does not have to store what it is given. This one stores Celsius whichever unit it is
handed, so the object keeps one value and shows it two ways.


In [12]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 1)

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = round((value - 32) * 5 / 9, 1)


reading = Reading(-4.1)
print("celsius", reading.celsius, "| fahrenheit", reading.fahrenheit)

reading.fahrenheit = 50
print("after reading.fahrenheit = 50: celsius", reading.celsius, "| fahrenheit", reading.fahrenheit)
print("what is stored:", vars(reading))


celsius -4.1 | fahrenheit 24.6
after reading.fahrenheit = 50: celsius 10.0 | fahrenheit 50.0
what is stored: {'celsius': 10.0}


Only `celsius` is stored, so the two can never disagree. Code that thinks in Fahrenheit reads and
writes `fahrenheit` as though it were an ordinary attribute, and the class converts in both
directions.

### How `@property` is built

`@property` is a decorator, so by the **Decorators** notebook's rule, `@property` above `def unit`
means `unit = property(unit)`. `property` is a class, and that line makes a `property` object holding
the function as its getter.

`@unit.setter` is a method of that object. It returns a **new** property with the same getter and
the decorated function as its setter, and the second `def unit` puts that new property under the
name `unit`. That is why both methods must share the attribute's name: the second replaces the first
with a property that has both.


In [13]:
held = vars(Station)["unit"]

print("Station.unit is a", type(held).__name__)
print("its getter:", held.fget.__name__, "| its setter:", held.fset.__name__)


Station.unit is a property
its getter: unit | its setter: unit


The same property can be built by hand, which is what the two decorators do for you.


In [14]:
def get_unit(self):
    return self._unit


def set_unit(self, value):
    if value not in ("C", "F"):
        raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
    self._unit = value


class ByHand:
    def __init__(self, unit="C"):
        self.unit = unit

    unit = property(get_unit, set_unit)


by_hand = ByHand("F")
print("read through the property:", by_hand.unit)

try:
    by_hand.unit = "kelvin"
except ValueError as error:
    print("and it checks:", error)


read through the property: F
and it checks: unit must be 'C' or 'F', not 'kelvin'


`property(get_unit, set_unit)` is still valid Python. The decorator form is what current code uses,
because it keeps the getter and the setter under the attribute's own name, next to each other.

### `cached_property`: worked out once

A computed property runs its method on every read. For something slow, `functools.cached_property`
runs it on the first read and stores the result on the object, so later reads cost nothing. It is the
property version of `functools.cache` from the **Decorators** notebook, and it comes with the same
condition.


In [15]:
class Archive:
    def __init__(self, readings):
        self.readings = readings

    @functools.cached_property
    def summary(self):
        print("  (working out the summary)")
        return {"n": len(self.readings), "coldest": min(self.readings)}


archive = Archive([-4.1, -2.6])
print(archive.summary)
print(archive.summary)
print("stored on the object:", "summary" in vars(archive))

archive.readings.append(-9.9)
print("after a colder reading:", archive.summary)

del archive.summary
print("after del archive.summary:", archive.summary)


  (working out the summary)
{'n': 2, 'coldest': -4.1}
{'n': 2, 'coldest': -4.1}
stored on the object: True
after a colder reading: {'n': 2, 'coldest': -4.1}
  (working out the summary)
after del archive.summary: {'n': 3, 'coldest': -9.9}


`working out the summary` printed once for two reads, and `summary` sits in the object's `vars`,
exactly where a plain attribute would. That is what makes it fast, and it is also why the read after
a colder reading was stale: the summary still said two readings and a coldest of `-4.1`.
`del archive.summary` threw the stored answer away, and the next read worked it out again.

This is the stored mean from the start of the notebook, on purpose. Use `cached_property` only for a
value that cannot change once the object is built.

### Putting it together: a station that shows either unit

One class using everything above: a computed, read-only mean, checked and tidied attributes, checks
that apply from construction, and one stored value shown two ways. The readings are stored once, in
Celsius, and `unit` decides only how they are shown.


In [16]:
class Station:
    """A station that stores Celsius and shows its readings in either unit."""

    def __init__(self, name, celsius, unit="C"):
        self.name = name
        self._celsius = list(celsius)
        self.unit = unit

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        value = value.strip()
        if not value:
            raise ValueError("a station needs a name")
        self._name = value

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value

    @property
    def readings(self):
        if self.unit == "C":
            return list(self._celsius)
        return [round(c * 9 / 5 + 32, 1) for c in self._celsius]

    @property
    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def add(self, celsius):
        self._celsius.append(celsius)


north = Station("  Tromso  ", [-4.1, -2.6])
print(f"{north.name!r} in {north.unit}: {north.readings}, mean {north.mean}")

north.unit = "F"
print(f"{north.name!r} in {north.unit}: {north.readings}, mean {north.mean}")

north.add(-5.2)
print(f"after add(-5.2):  {north.readings}, mean {north.mean}")


'Tromso' in C: [-4.1, -2.6], mean -3.35
'Tromso' in F: [24.6, 27.3], mean 25.95
after add(-5.2):  [24.6, 27.3, 22.6], mean 24.83


The name arrived with spaces and was stored without them. Switching `unit` to `'F'` converted every
reading and the mean without any conversion method being called, because `readings` and `mean` are
both worked out when they are read. `add` takes Celsius, and the mean in Fahrenheit reflected it
immediately.

Now the rules.


In [17]:
try:
    north.mean = 0
except AttributeError as error:
    print("north.mean = 0         ", error)

try:
    north.unit = "kelvin"
except ValueError as error:
    print("north.unit = 'kelvin'  ", error)

try:
    Station("   ", [1.0])
except ValueError as error:
    print("Station('   ', [1.0])  ", error)

print()
print("what is actually stored:", vars(north))


north.mean = 0          property 'mean' of 'Station' object has no setter
north.unit = 'kelvin'   unit must be 'C' or 'F', not 'kelvin'
Station('   ', [1.0])   a station needs a name

what is actually stored: {'_name': 'Tromso', '_celsius': [-4.1, -2.6, -5.2], '_unit': 'F'}


Every rule held. A calculated value could not be assigned, a bad unit was refused on the line that
tried it, and a station could not even be created without a name, because `__init__` assigns through
the properties. What is stored is three underscore attributes, and neither `mean` nor `readings` is
among them.

### Where each part came from

| In `Station` | What it relies on | The section that showed it |
|---|---|---|
| `mean`, with no setter | a computed property is current, and read-only | A computed attribute, and Read-only by default |
| `unit = "kelvin"` refused | a setter sees every assignment | Checking a value on the way in |
| `Station("   ", ...)` refused | `__init__` assigns through the properties | Checking in `__init__` too |
| `_name`, `_unit` and `_celsius` | the underscore marks what is internal | The underscore convention |
| `name` tidied before it is stored | a setter can store something other than it was given | A property that converts |
| `readings` shown in either unit | one stored value, two views | A property that converts |
| each getter and setter sharing a name | `@name.setter` replaces the property with one that has both | How `@property` is built |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/07-properties-solutions.ipynb).

**1.** Give a small `Station` class a `spread` property, the highest reading minus the lowest. Read
it without parentheses, then add a reading and show that `spread` is already up to date.


In [18]:
# your code here


**2.** Make `name` a checked property whose setter strips surrounding spaces and raises `ValueError`
for a name that is empty after stripping. Show a name with spaces being tidied, and an empty one
being rejected.


In [19]:
# your code here


**3.** Try to assign a number to `spread`. Read the error, then say in a comment why it is correct
that this fails.


In [20]:
# your code here


**4.** Write a `Reading` class that stores only `celsius`, with a `fahrenheit` property that has a
getter and a setter which converts. Set `fahrenheit` to `212` and print `celsius`.


In [21]:
# your code here


**5.** Give `Station` a `readings` property whose setter raises `ValueError` if any reading is below
`-90` or above `60`, and make `__init__` assign through it. Show a valid list accepted, a list
containing `999.0` rejected, and a station that cannot be created with one.


In [22]:
# your code here


**6.** Give an `Archive` class a `functools.cached_property` called `summary`. Show that it is worked
out once across two reads, then add a reading and show that it is stale. Say in a comment when that
is acceptable.


In [23]:
# your code here


## Common errors

### TypeError: calling a property as though it were a method

A property is read without parentheses. Adding them calls whatever the property gave back.


In [24]:
class Station:
    def __init__(self, readings):
        self.readings = readings

    @property
    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)


north = Station([-4.1, -2.6])
north.mean()


TypeError: 'float' object is not callable

`north.mean` ran the property and produced `-3.35`, a `float`, and then `()` tried to call that
number. The message names the type of the value rather than the property, so it can take a moment to
connect. When `'float' object is not callable` appears on a line that reads an attribute, that
attribute is almost always a property.

This is the reverse of the missing-parentheses error in the **Methods** notebook, where leaving them
off a method gave back the method itself.

### AttributeError: no setter, when you wrote one

The setter has to be defined under the same name as the property. Name it anything else and it
becomes a separate property, and the original stays read-only.


In [25]:
class WrongName:
    def __init__(self, unit):
        self._unit = unit

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def set_unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value


station = WrongName("C")
station.unit = "F"


AttributeError: property 'unit' of 'WrongName' object has no setter

`@unit.setter` built a new property with both methods, and `def set_unit` put it under the name
`set_unit`. `unit` still holds the original property, the one with no setter. The class now has two
properties, and neither is the one you meant.


In [26]:
print(sorted(name for name in vars(WrongName) if not name.startswith("__")))


['set_unit', 'unit']


Both methods must be named after the attribute. It reads oddly the first time, and it is exactly what
makes the replacement described in How `@property` is built work.

### The setter that assigns to itself

Inside the `unit` setter, `self.unit = value` is an assignment to the property, so it calls the
setter, which makes the same assignment again, without end. A getter that returns `self.unit` does
the same. The value has to go to `self._unit`.

Left alone, this does not stop on its own. A plain Python script ends with
`RecursionError: maximum recursion depth exceeded`. In a notebook the kernel can die instead, taking
every variable with it, so the cell below counts the setter's calls and stops itself after five.


In [27]:
class Loops:
    calls = 0

    def __init__(self, unit):
        self.unit = unit

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def unit(self, value):
        Loops.calls += 1
        print(f"  setter call {Loops.calls}")
        if Loops.calls == 5:
            raise RecursionError("stopped by hand after 5 calls")
        self.unit = value  # the bug: this calls the setter again


try:
    Loops("C")
except RecursionError as error:
    print("RecursionError:", error)


  setter call 1
  setter call 2
  setter call 3
  setter call 4
  setter call 5
RecursionError: stopped by hand after 5 calls


Each call to the setter began another call to the setter, and none of them ever reached a line that
stored anything. The fix is one character, `self._unit = value`, which stores the value instead of
assigning to the property.

### The quiet one: `__init__` stores the value past the check

An `__init__` that writes to the underscore attribute directly never calls the setter.


In [28]:
class Unchecked:
    def __init__(self, unit="C"):
        self._unit = unit

    @property
    def unit(self):
        return self._unit

    @unit.setter
    def unit(self, value):
        if value not in ("C", "F"):
            raise ValueError(f"unit must be 'C' or 'F', not {value!r}")
        self._unit = value


station = Unchecked("kelvin")
print("created with unit:", station.unit)


created with unit: kelvin


No error. The setter would have refused `'kelvin'`, and it never ran, because `self._unit = unit` went
around it. Every later assignment is checked, so a test that creates a station and then changes its
unit passes, while every station created with a bad unit keeps it for good.

Assign through the property in `__init__`, as `self.unit = unit`. The only line in the class that
writes `self._unit` should be the one inside the setter.


## Recap

- Start with plain attributes. A property can replace one later without any calling code changing.
- `@property` makes reading `station.mean` run a method, with no parentheses.
- A computed property is worked out when it is read, so it is always current.
- A property with no setter cannot be assigned, which suits a calculated value.
- `@name.setter` on a second method with the same name makes assignment run that method.
- A setter sees every assignment, so a check placed there cannot be forgotten by a caller.
- Assign through the property in `__init__`, so construction is checked too.
- The value itself lives in an underscore attribute, which is internal by convention only.
- A setter that assigns to its own property calls itself without end. Store in the underscore name.
- A setter can store something other than it was given, so one value can be shown two ways.
- `@property` builds a `property` object, and `@name.setter` returns a new one holding both methods.
- `cached_property` works a value out once and stores it, so it goes stale if its inputs change.
- Calling a property, as `station.mean()`, calls the value it gave back.


## What is next

The **Class and Static Methods** notebook. Every method so far has been called on an object and handed
it as `self`. Some methods belong to the class instead: building a `Station` from one line of a CSV
file, before any station exists, or a conversion that needs no station at all. `@classmethod` and
`@staticmethod` are the two decorators for those.


---

&#8592; **Previous:** [Decorators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/06-decorators.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
